This code uses models hosted on GitHub and uses Langchain ecosystem (LangSmith, LangServe etc)

STEP 0: Setup the LLM infrastructure

In [ ]:
from dotenv import load_dotenv
load_dotenv(override=True)

import os

# This setup is required to log events into Langsmith
os.environ["API_HOST"] = os.getenv("API_HOST", "github")
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY", "")
os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGCHAIN_PROJECT", "")
os.environ["LANGCHAIN_ENDPOINT"] = os.getenv("LANGCHAIN_ENDPOINT")
os.environ["LANGCHAIN_TRACING_V2"] = os.getenv("LANGCHAIN_TRACING_V2")


API_HOST = os.getenv("API_HOST", "github")


print(os.environ["LANGCHAIN_API_KEY"][:5])
print(os.environ["LANGCHAIN_PROJECT"])
print(os.environ["LANGCHAIN_TRACING_V2"])

In [ ]:
from langchain_openai import ChatOpenAI

if API_HOST == "github":
    print("Using GitHub models...and model:", os.getenv("GITHUB_MODEL", "openai/gpt-4o"))
    llm = ChatOpenAI(
        model_name=os.getenv("GITHUB_MODEL", "openai/gpt-4o"),
        openai_api_base="https://models.github.ai/inference",
        openai_api_key=os.environ["GITHUB_TOKEN"],
    )
elif API_HOST == "ollama":
    print("Using Ollama model on local...")
    llm = ChatOpenAI(
        model_name=os.getenv("OLLAMA_MODEL", "mistral"),
        openai_api_base=os.environ["OLLAMA_ENDPOINT"],
        openai_api_key="nokeyneeded",
    )
"""
from langsmith import Client
client = Client()
print(client)

client.create_project(
    project_name="my-langchain-project",
    description="LangChain + GitHub Models tracing"
)

print(client.read_project(project_name=os.environ["LANGCHAIN_PROJECT"]))
"""



In [ ]:

# Testing using Langchain
response = llm.invoke("what is BERT")
print(response)

Using Chat Prompt Templates

In [ ]:
# Using prompt templates and chaining. This is also known as Langchain Expression Language (LCEL)
from langchain_core.prompts import ChatPromptTemplate


prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are an expert AI Engineer. Provide me answers based on the questions.."), 
        ("user", "{input}")
    ]
)

#print(prompt.invoke({"input":"What is LSTM?"}))

chain = prompt | llm # This is LCEL
response = chain.invoke({"input": "What is LSTM?"})
print(f"Response from {API_HOST}: \n")
print(response.content) 





Using StrOutput Parser

In [ ]:
# Using StrOutputParser
from langchain_core.output_parsers import StrOutputParser

output_parser = StrOutputParser()
chain = prompt | llm | output_parser # This is LCEL
response = chain.invoke({"input": "What is LSTM?"})
print(response) # This provides just the content

Using System and Human Messages

In [ ]:
# Using System and Human Messages
from langchain_core.messages import HumanMessage, SystemMessage

messages = [
SystemMessage(content="Translate from English to Marathi"),
HumanMessage(content="How are you")]

response = llm.invoke(messages)

# Using StrOutputParser
output_parser.invoke(response)


Using Message Histories

In [51]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store = {} # this data stricture will store the session ids and chat message history

# This function will be used to track the history per session as there can be multiple chat sessions
def get_session_history(session_id:str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    
    return store[session_id]

with_message_history = RunnableWithMessageHistory(llm,get_session_history)

config={"configurable":{"session_id":"chat_1"}} # hardcoded the chat session id

response = with_message_history.invoke(
    [HumanMessage(content="Hi, I am Satyaki and I am a data architect")],
    config=config
    )

response.content


'Hi Satyaki! It’s great to meet you. As a data architect, you must work on designing and managing data systems and structures. What specific areas or projects are you currently focused on?'

In [52]:
# Let's ask the LLM for the same config i.e. session id
response = with_message_history.invoke(
    [HumanMessage(content="What's my name?")],
    config=config
    )

response.content

'Your name is Satyaki. How can I assist you today?'

In [53]:
# Let's change teh config --> change session id
config1={"configurable":{"session_id":"chat_2"}} # hardcoded the chat session id


response = with_message_history.invoke(
    [HumanMessage(content="What's my name?")],
    config=config1
    )

response.content

"I'm sorry, but I don't know your name. If you'd like to share it, feel free!"

In [ ]:
# Using Message Place Holder

from langchain_core.prompts import MessagesPlaceholder
from langchain_core.messages import HumanMessage
from langchain_core.runnables.history import RunnableWithMessageHistory


prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are an expert AI Engineer. Provide me answers based on the questions.."), 
        MessagesPlaceholder(variable_name="messages") #created a variable to store all user input messages
    ]
)

#print(prompt.invoke({"input":"What is LSTM?"}))

chain = prompt | llm # This is LCEL
response = chain.invoke({"messages":[HumanMessage(content="My name is Satyaki")]})
print(f"Response from {API_HOST}: \n")
print(response.content) 


Response from github: 

Nice to meet you, Satyaki! How can I assist you today?


In [72]:
# Now lets change to pass multiple inputs.

prompt_2 = ChatPromptTemplate.from_messages(
    [
        ("system", "You are an expert Language Traslator for {language}. Translate for the given language"), 
        MessagesPlaceholder(variable_name="messages"), #created a variable to store all user input messages
    ]
)


chain_2 = prompt_2 | llm
config2={"configurable":{"session_id":"chat_3"}} # hardcoded the chat session id

response = chain_2.invoke({"messages":[HumanMessage(content="this is Satyaki")],"language":"Hindi"})


with_message_history = RunnableWithMessageHistory(chain_2,get_session_history,input_messages_key="messages")
config4={"configurable":{"session_id":"chat_4"}} # hardcoded the chat session id

response = with_message_history.invoke(
    {
        "messages": [HumanMessage(content="This is Satyaki")],
        "language": "Hindi"        
    },
    config=config4
)
response


AIMessage(content='यह सत्यकी है।', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 6, 'prompt_tokens': 67, 'total_tokens': 73, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_f97eff32c5', 'id': 'chatcmpl-CsmQJ1XZXzhoOR3m1ITpXAdii17Zv', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019b73a7-0df1-7921-8f41-28034d689009-0', usage_metadata={'input_tokens': 67, 'output_tokens': 6, 'total_tokens': 73, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

Manage Conversation History

In [77]:
from langchain_core.messages import SystemMessage, trim_messages, AIMessage
from langchain_core.runnables import RunnablePassthrough
from operator import itemgetter

prompt_2 = ChatPromptTemplate.from_messages(
    [
        ("system", "You are an assistant. Translate for the given {language}"), 
        MessagesPlaceholder(variable_name="messages"), #created a variable to store all user input messages
    ]
)


trimmer = trim_messages(
    max_tokens=45,
    strategy="last",
    token_counter=llm,
    include_system=True,
    allow_partial=False,
    start_on="human"
)
messages = [
    SystemMessage(content="you're a good assistant"),
    HumanMessage(content="hi! I'm bob"),
    AIMessage(content="hi!"),
    HumanMessage(content="I like vanilla ice cream"),
    AIMessage(content="nice"),
    HumanMessage(content="whats 2 + 2"),
    AIMessage(content="4"),
    HumanMessage(content="thanks"),
    AIMessage(content="no problem!"),
    HumanMessage(content="having fun?"),
    AIMessage(content="yes!"),
]
trimmer.invoke(messages)


chain = (
            RunnablePassthrough.assign(messages=itemgetter("messages") | trimmer) | prompt_2 | llm

)

response=chain.invoke(
    {
    "messages":messages + [HumanMessage(content="What ice cream do i like")],
    "language":"English"
    }
)
response.content

'I’m not sure what ice cream you like, but some popular flavors include chocolate, vanilla, strawberry, and mint chocolate chip. Do you have a favorite?'

In [64]:
## Lets wrap this in the MEssage History
with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages",
)
config={"configurable":{"session_id":"chat5"}}

In [65]:
response = with_message_history.invoke(
    {
        "messages": messages + [HumanMessage(content="whats my name?")],
        "language": "English",
    },
    config=config,
)

response.content

'Could you please clarify your question about "messages" or "language"? Are you inquiring about messaging systems, programming languages, language processing, or something else? Let me know how I can assist you!'

STEP 1: Loading Documents

In [ ]:
# Text Loader
from langchain_community.document_loaders import TextLoader
loader = TextLoader("speech.txt")
text_documents = loader.load() # loads the text file into a list of Document objects
text_documents
print(f"\nLoaded {len(text_documents)} document(s) from speech.txt")


In [ ]:
# PDF Loader
from langchain_community.document_loaders import PyPDFLoader
from rich import print

pdf_loader = PyPDFLoader("attention.pdf")
pdf_documents = pdf_loader.load() # loads the PDF file into a list of Document objects
pdf_documents
#print(f"\nLoaded {len(pdf_documents)} document(s) from attention.pdf")

In [ ]:
# Web based loader
from langchain_community.document_loaders import WebBaseLoader
import bs4

web_loader = WebBaseLoader(web_paths=("https://en.wikipedia.org/wiki/Attention_(machine_learning)",), 
                           bs_kwargs=dict(parse_only=bs4.SoupStrainer(class_=("post_content","post_title","post_header"))))
web_loader.load()

STEP 2: Text Splitting into Chunks

In [ ]:
## How to recursively split text by charecters . Eg using PDF files

from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)

final_pdf = text_splitter.split_documents(pdf_documents) # since the pdf_documents are of Document type
final_pdf
print(f"Total chunks created: {len(final_pdf)} as Document Type")

In [ ]:
## Lets read a txt file and split into chunks
# You can also use PDF Loader feature instead.

with open("speech.txt") as f:
    speech = f.read()

speech
final_text = text_splitter.create_documents([speech])
print(f"Total chunks created: {len(final_text)} as Document Type")

In [ ]:
# Let's do using CharecterTextSplitter another way
from langchain_text_splitters import CharacterTextSplitter

char_text_splitter = CharacterTextSplitter(separator="\n\n", chunk_size=500, chunk_overlap=50)

final_text_using_char = char_text_splitter.split_documents(text_documents) # text_documents using TextLoader
final_text_using_char

In [ ]:
## Splitting HTML Documents
from langchain_text_splitters import HTMLHeaderTextSplitter

html_string="""
<!DOCTYPE html>
<html>
<body>
    <div>
        <h1>Foo</h1>
        <p>Some intro text about Foo.</p>
        <div>
            <h2>Bar main section</h2>
            <p>Some intro text about Bar.</p>
            <h3>Bar subsection 1</h3>
            <p>Some text about the first subtopic of Bar.</p>
            <h3>Bar subsection 2</h3>
            <p>Some text about the second subtopic of Bar.</p>
        </div>
        <div>
            <h2>Baz</h2>
            <p>Some text about Baz</p>
        </div>
        <br>
        <p>Some concluding text about Foo</p>
    </div>
</body>
</html>
"""

html_header_split_on = [
    ("h1","Header 1"),
    ("h2","Header 2"),
    ("h3","Header 3")
]

html_splitter = HTMLHeaderTextSplitter(headers_to_split_on=html_header_split_on)
html_header_splits = html_splitter.split_text(html_string)
html_header_splits


In [ ]:
## Splitting JSON text
import json
import requests

json_data = requests.get("https://api.smith.langchain.com/openapi.json").json()

from langchain_text_splitters import RecursiveJsonSplitter

json_splitter = RecursiveJsonSplitter(max_chunk_size=300)
json_chunks = json_splitter.split_json(json_data)

# We can also create this as Document Type
json_docs = json_splitter.create_documents(texts=[json_data])
json_docs
# Just get the content
json_text = json_splitter.split_text(json_data=json_data)
json_text


STEP 3: Vector Embeddings

In [ ]:
## Using OpenAI Embeddings (read Open AI documentation)

from langchain_openai import OpenAIEmbeddings
#embeddings = OpenAIEmbeddings(model='text-embedding-3-large') """This code will work if you have an OpenAI account and API Key"""

embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-small",
    api_key=os.environ["GITHUB_TOKEN"],  # Your GitHub PAT
    base_url="https://models.inference.ai.azure.com")

text="how are you"

query_result = embedding_model.embed_query(text)
query_result



In [ ]:

"""Another way - we will use Azure AI Inference SDK which is hosted on GitHub Infra
from azure.ai.inference import EmbeddingsClient
from azure.core.credentials import AzureKeyCredential

# Point the client to the GitHub Models endpoint
client = EmbeddingsClient(
    endpoint="https://models.inference.ai.azure.com",
    credential=AzureKeyCredential(os.environ["GITHUB_TOKEN"]))

# Request embeddings
embeddings = client.embed(
    input=["Hello, world!", "GitHub Models are great for prototyping."],
    model="text-embedding-3-small"
)
embeddings['data']
"""



In [ ]:
## Vector Embeddings and Vector Store (using Chroma)
from langchain_community.vectorstores import Chroma

#final_text: using the text converted into chunks in previous steps.
#collection_name: the default using langchain is "langchain" which uses 384 dimensions
db = Chroma.from_documents(final_text, embedding_model,collection_name="github_openai_embeddings") 

In [ ]:
## Retrieved the results by querying ChromaDB
query = 'It will be all the easier for us to conduct ourselves as belligerents'

retrieved_results = db.similarity_search(query=query)
retrieved_results

In [ ]:
## Let's using Ollama (local)

from langchain_community.embeddings import OllamaEmbeddings #this is deprecated
#from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(model = 'mistral') # by default it uses llama2
r1 = embeddings.embed_documents(['thsi is a test', 'this is a seconds test'])
r1


In [ ]:
## Using Hugging Face
HF_TOKEN = os.getenv('HF_TOKEN')

from langchain_huggingface import HuggingFaceEmbeddings
embedding_model = HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2')

query = "how are you"

result = embedding_model.embed_query(query)
result